# Stage 7: State-Level Cybercrime Volume Prediction
**Project**: Cyber Crime Analytics for National Security  
**Data Source**: Longitudinal State-Level Series (2018–2022) (`trend_2018_2022.csv`)  
**Methodological Position**: Temporal Panel Lag Prediction with Out-of-Sample Held-Out Evaluation ($N = 106$)

---

## 1. Objective & Analytical Scope

### Analytical Problem Definition:
The objective of this stage is to construct and evaluate **defensible predictive models** for state-level aggregate cybercrime volume across time.

### Methodological Guardrails & Zero-Leakage Protocol:
1. **Tautological / Part-Whole Leakage Prohibition**: Regressing aggregate crime volume on its simultaneous component subcategories (e.g., predicting total cases from IT Act offences or cheating subcategories) is a mathematical identity and academically invalid. In this analysis, **no simultaneous crime breakdowns or motive counts are used as predictors**.
2. **Strict Chronological Separation**: All predictive features are constructed strictly from historical preceding years ($t-1$ and $t-2$). The models are trained on historical target years (2020 and 2021, $N_{\text{train}} = 70$) and evaluated on a **strictly held-out future test year** (2022, $N_{\text{test}} = 36$).
3. **Baselines First**: Simple, academically standard baselines (Naive Persistent Lag-1 and Historical Moving Average) are evaluated first to establish a rigorous performance benchmark before introducing machine learning models.
4. **Neutral Academic Interpretation**: Results describe longitudinal scale inertia and predictive error distributions. They do not imply causality, state-level criminality, or operational deployment readiness.


In [1]:
# Setup environment, paths, and imports
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Stage 7 prediction module routines
from src.prediction import (
    load_and_construct_panel_dataset,
    run_leakage_audit,
    train_and_evaluate_models,
    plot_actual_vs_predicted,
    plot_model_comparison,
    plot_residuals_by_state,
    plot_historical_trajectory_forecast,
    export_prediction_outputs
)

print(f"Working Directory: {project_root}")
print("Stage 7 Prediction module loaded successfully.")


Working Directory: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics
Stage 7 Prediction module loaded successfully.


---
## Section B — Data Inspection & Lagged Panel Feature Engineering

We load the historical 2018–2022 longitudinal series and construct panel observations with:
- **`lag_1`**: Cybercrime cases in year $t-1$ (Immediate baseline scale)
- **`lag_2`**: Cybercrime cases in year $t-2$ (Prior momentum level)
- **`lag_diff`**: Change between $t-2$ and $t-1$ ($y_{t-1} - y_{t-2}$)
- **`lag_growth_rate`**: Historical annual rate of change
- **`log_lag_1` & `log_lag_2`**: $\log(1 + y)$ transformed features to stabilize extreme positive skewness


In [2]:
# Load historical data and construct chronological panel dataset
trend_path = project_root / 'data' / 'processed' / 'trend_2018_2022.csv'
panel_df = load_and_construct_panel_dataset(trend_path)

print(f"Constructed Panel Dataset: {panel_df.shape[0]} Observations across {panel_df['state_name'].nunique()} Jurisdictions")
print("\n--- Split Distribution by Target Year ---")
display(pd.crosstab(panel_df['target_year'], panel_df['split'], margins=True))

print("\n--- Panel Dataset Sample (First 5 Rows) ---")
display(panel_df.head())


Constructed Panel Dataset: 106 Observations across 36 Jurisdictions

--- Split Distribution by Target Year ---

--- Panel Dataset Sample (First 5 Rows) ---


split,test,train,All
target_year,,,
2020,0,35,35
2021,0,35,35
2022,36,0,36
All,36,70,106


,state_name,target_year,split,lag_1,lag_2,lag_diff,lag_growth_rate,lag_mean_2yr,log_lag_1,log_lag_2,target_actual
0,Andaman and Nicobar Islands,2020,train,2.0,7.0,-5.0,-0.625000,4.5,1.098612,2.079442,5.0
1,Andhra Pradesh,2020,train,1886.0,1207.0,679.0,0.562086,1546.5,7.542744,7.096721,1899.0
2,Arunachal Pradesh,2020,train,8.0,7.0,1.0,0.125000,7.5,2.197225,2.079442,30.0
3,Assam,2020,train,2231.0,2022.0,209.0,0.103312,2126.5,7.710653,7.612337,3530.0
4,Bihar,2020,train,1050.0,374.0,676.0,1.802667,712.0,6.957497,5.926926,1512.0


---
## Section C — Automated Target Leakage & Chronology Audit

Before fitting any models, we execute an automated 5-point leakage audit to verify that:
1. The target variable is strictly excluded from all predictor matrices.
2. The training window (2020–2021) strictly precedes the test window (2022).
3. No overlapping (state, year) observation tuples exist across splits.
4. No simultaneous 2023 detailed category breakdowns enter historical training.
5. Zero missing / null values exist in features or targets.


In [3]:
# Execute automated leakage audit
audit_df = run_leakage_audit(panel_df)
print("--- Stage 7 Automated Leakage & Methodology Audit ---")
display(audit_df[['check_id', 'check_name', 'status', 'description', 'details']])


--- Stage 7 Automated Leakage & Methodology Audit ---


,check_id,check_name,status,description,details
0,LEAK-01,Predictor-Target Disjunction,PASSED,Target variable target_actual is strictly excluded from predictor matrices.,Predictor set contains only historical lagged indicators.
1,LEAK-02,Chronological Split Validity,PASSED,Training window strictly precedes held-out test window (Train <= 2021 < Test = 2022).,"Train years: [2020, 2021], Test years: [2022]."
2,LEAK-03,Observation Independence,PASSED,"Zero overlap in (state_name, target_year) tuples across train and test partitions.",Intersection count = 0.
3,LEAK-04,No 2023 Detailed Component Leakage,PASSED,"No 2023 category breakdowns, IT Act subtotals, or motive counts used.",Dataset constructed purely from historical 2018-2022 state annual series.
4,LEAK-05,Null Value Verification,PASSED,Zero NaN / null values in panel features and target.,Total null count = 0.


---
## Section D — Model Training & Out-of-Sample Test Evaluation

We train seven distinct baseline and machine learning models on the training set (2020–2021, $N_{\text{train}} = 70$) and evaluate their predictive accuracy on the held-out test set (2022, $N_{\text{test}} = 36$):
1. **Naive Persistent (Lag-1)**: Baseline predicting $\hat{y}_t = y_{t-1}$.
2. **Historical 2-Year Moving Average**: Baseline predicting $\hat{y}_t = (y_{t-1} + y_{t-2})/2$.
3. **Linear Regression (OLS, Raw)**: Standard Ordinary Least Squares regression on raw historical counts.
4. **Ridge Regression (L2 Regularized)**: Linear model with L2 regularization penalty $(\alpha = 1.0)$.
5. **Log-Linear Regression (Log OLS)**: Linear regression on $\log(1+y)$ transformed features with exponential back-transformation.
6. **Decision Tree Regressor**: Constrained tree model (depth = 3) to prevent overfitting.
7. **Random Forest Regressor**: Ensemble of 100 shallow trees (depth = 3, random_state = 42).


In [4]:
# Train models and evaluate on held-out 2022 test partition
results_df, predictions_df, trained_models = train_and_evaluate_models(panel_df)

print("--- Model Evaluation Summary (Held-out 2022 Test Set, N = 36) ---")
display(results_df[['model_name', 'model_type', 'feature_set', 'mae', 'rmse', 'r2', 'median_ae', 'description']])


--- Model Evaluation Summary (Held-out 2022 Test Set, N = 36) ---


,model_name,model_type,feature_set,mae,rmse,r2,median_ae,description
0,Naive Persistent (Lag-1),Baseline,lag_1 (Year t-1),564.75,1340.75,0.8625,61.50,Assumes cybercrime volume remains identical to the immediately preceding year.
1,Historical 2-Year Moving Average,Baseline,"lag_1, lag_2 Mean",563.04,1523.73,0.8224,57.00,Predicts the arithmetic mean of the preceding two years.
2,"Linear Regression (OLS, Raw)",Linear Model,"lag_1, lag_2",776.08,1680.13,0.7840,205.07,Ordinary Least Squares regression on raw historical counts.
3,Ridge Regression (L2 Regularized),Linear Model,"lag_1, lag_2",776.08,1680.13,0.7840,205.07,Linear regression with L2 regularization penalty (alpha=1.0).
4,Log-Linear Regression (Log OLS),Log-Linear Model,"log_lag_1, log_lag_2",479.37,1143.46,0.9000,69.85,Linear regression on log1p counts with exponential back-transformation.
5,Decision Tree Regressor,Tree Model,"lag_1, lag_2",608.42,1571.47,0.8111,100.86,Single decision tree with constrained depth (max_depth=3) to prevent overfitting.
6,Random Forest Regressor,Ensemble Model,"lag_1, lag_2",582.24,1559.55,0.8139,58.47,"Random forest ensemble of 100 shallow trees (max_depth=3, random_state=42)."


---
## Section E — Diagnostic Visualizations & Error Analysis


In [5]:
# Figure 20: Actual vs. Predicted Parity Plots
fig_parity = plot_actual_vs_predicted(
    predictions_df,
    save_path=str(project_root / 'outputs' / 'figures' / '20_prediction_actual_vs_predicted.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [6]:
# Figure 21: Model Performance Comparison Bar Chart
fig_comp = plot_model_comparison(
    results_df,
    save_path=str(project_root / 'outputs' / 'figures' / '21_prediction_model_comparison.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [7]:
# Figure 22: State-by-State Prediction Residuals (Log-Linear Model)
fig_residuals = plot_residuals_by_state(
    predictions_df,
    save_path=str(project_root / 'outputs' / 'figures' / '22_prediction_residuals_by_state.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [8]:
# Figure 23: Historical Trajectories & Out-of-Sample Forecasts
fig_traj = plot_historical_trajectory_forecast(
    trend_path=str(trend_path),
    predictions_df=predictions_df,
    save_path=str(project_root / 'outputs' / 'figures' / '23_historical_trajectory_forecast.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section F — State-by-State 2022 Prediction Breakdown


In [9]:
# Display complete state-by-state 2022 prediction and error table
print("--- State-by-State 2022 Prediction Breakdown (Actual vs Forecasts) ---")
display(predictions_df[['state_name', 'actual_2022', 'pred_naive_lag1', 'pred_log_linear', 'pred_random_forest',
                        'error_naive', 'error_log_linear', 'error_rf']])


--- State-by-State 2022 Prediction Breakdown (Actual vs Forecasts) ---


,state_name,actual_2022,pred_naive_lag1,pred_log_linear,pred_random_forest,error_naive,error_log_linear,error_rf
0,Andaman and Nicobar Islands,28,8,10,59,-20,-18,31
1,Andhra Pradesh,2341,1875,2489,1899,-466,148,-442
2,Arunachal Pradesh,14,47,55,59,33,41,45
3,Assam,1733,4846,5730,6303,3113,3997,4570
4,Bihar,1621,1413,1913,1578,-208,292,-43
5,Chandigarh,27,15,22,59,-12,-5,32
6,Chhattisgarh,439,352,444,557,-87,5,118
7,Dadra and Nagar Haveli and Daman and Diu,5,5,6,59,0,1,54
8,Delhi,685,356,371,514,-329,-314,-171
9,Goa,90,36,51,59,-54,-39,-31


---
## Section G — Analytical Interpretation & Methodological Findings

### 1. Model Performance & Scale Inertia:
- **Strong Baseline Performance**: The Naive Persistent Baseline ($y_{t-1}$) achieves a high $R^2 = 0.8625$ ($	ext{MAE} = 564.75$ cases), demonstrating substantial longitudinal scale persistence across Indian State/UT jurisdictions.
- **Superiority of Log-Linear Modeling**: The **Log-Linear Regression model** achieves the highest overall accuracy ($	ext{MAE} = 479.37$ cases, $	ext{RMSE} = 1,143.46$, $R^2 = 0.9000$). By modeling proportional growth on the log scale, it effectively stabilizes variance across both high-volume states (e.g., Karnataka, Telangana) and small Union Territories (e.g., Lakshadweep, Ladakh).
- **Raw Linear Regression Limitations**: Unregularized OLS on raw counts is disproportionately influenced by the extreme upper tail, leading to a higher $	ext{MAE} = 776.08$ cases.

### 2. Residual Distribution & Prediction Outliers:
- **Rapid Acceleration Hubs**: In states experiencing sudden exponential surges in registration (e.g., Telangana rising from 10,303 in 2021 to 15,297 in 2022; Karnataka rising from 8,136 to 12,556), historical lag models tend to underpredict the surge.
- **Registration Policy Shifts**: In jurisdictions experiencing sudden drops after preceding spikes (e.g., Assam dropping from 4,846 in 2021 to 1,733 in 2022), models overpredict based on historical momentum.


In [10]:
# Export Tables and Serialized Model Artifacts
exported_paths = export_prediction_outputs(
    panel_df=panel_df,
    results_df=results_df,
    predictions_df=predictions_df,
    audit_df=audit_df,
    trained_models=trained_models,
    output_dir=str(project_root / 'outputs' / 'tables'),
    models_dir=str(project_root / 'outputs' / 'models')
)

print("Exported Stage 7 Prediction Output Files:")
for k, v in exported_paths.items():
    p = Path(v)
    print(f"  - {k:<25}: {p.name} ({p.stat().st_size:,} bytes)")


Exported Stage 7 Prediction Output Files:
  - dataset                  : prediction_dataset.csv (11,085 bytes)
  - results                  : prediction_results.csv (1,237 bytes)
  - predictions              : prediction_actual_vs_predicted.csv (2,074 bytes)
  - audit                    : prediction_leakage_audit.csv (869 bytes)
  - model_linear_regression_ols,_raw: linear_regression_ols,_raw.pkl (502 bytes)
  - model_ridge_regression_l2_regularized: ridge_regression_l2_regularized.pkl (457 bytes)
  - model_log_linear_regression_log_ols: log_linear_regression_log_ols.pkl (502 bytes)
  - model_decision_tree_regressor: decision_tree_regressor.pkl (2,096 bytes)
  - model_random_forest_regressor: random_forest_regressor.pkl (129,952 bytes)


---
## Section H — Methodological Limitations & Analytical Boundaries

### Explicit Constraints:
1. **Limited Temporal Depth**: The available series covers only 5 annual points (2018–2022), restricting training to $N_{\text{train}} = 70$ panel observations.
2. **Aggregate-Level Only**: Predictions apply to annual state-level case totals. Incident-level or sub-annual prediction is unsupported by the dataset.
3. **Reporting Shifts vs. True Crime**: Variations in annual case counts reflect shifts in police registration practices, reporting portals, and cybercrime cell establishment rather than pure shifts in criminal propensity.
4. **Non-Causal Nature**: These models capture longitudinal correlation and scale inertia. They do not identify causal socio-economic or technological drivers.
